In [11]:
import requests

URL = "https://books.toscrape.com/catalogue/page-1.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (sebastien.hetsch.data@gmail.com)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

print("status   :", r.status_code)
print("longueur :", len(r.text), "caracteres")
print("debut    :", r.text[:60])


status   : 200
longueur : 50449 caracteres
debut    : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [12]:
print("le prix est present  :", "£51.77" in r.text)
print("le titre est present :", "A Light in the Attic" in r.text)


le prix est present  : True
le titre est present : True


In [13]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")
fiches = soup.select("article.product_pod")
print("fiches trouvees :", len(fiches))


fiches trouvees : 20


In [14]:
fiche = fiches[0]

prix = fiche.select_one("p.price_color").get_text(strip=True)   # contenu
lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]                                         # attribut
href = lien_h3["href"]                                           # attribut

print("prix  :", prix)
print("titre :", titre)
print("href  :", href)


prix  : £51.77
titre : A Light in the Attic
href  : a-light-in-the-attic_1000/index.html


In [15]:
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

classes = fiche.select_one("p.star-rating")["class"]
print("classes :", classes)
print("note    :", NOTES[classes[1]])


classes : ['star-rating', 'Three']
note    : 3


In [16]:
from urllib.parse import urljoin

print(urljoin(URL, href))


https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [17]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }

print(lire_fiche(fiches[0], URL))


{'titre': 'A Light in the Attic', 'prix': '£51.77', 'dispo': 'In stock', 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


In [18]:
suivant = soup.select_one("li.next a")
print(suivant["href"] if suivant else "pas de page suivante")


page-2.html


In [19]:
import time

def collecter_toutes_les_pages(url_depart, pages_max=3):
    url = url_depart
    livres = []
    page = 0
    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"  page {page} : status {r.status_code}, on s'arrete")
            break
        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]
        print(f"  page {page} : {len(fiches)} livres  (total {len(livres)})")
        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None
        time.sleep(1)
    return livres


In [20]:
livres = collecter_toutes_les_pages("https://books.toscrape.com/catalogue/page-1.html")
print("collecte :", len(livres), "livres")


  page 1 : 20 livres  (total 20)
  page 2 : 20 livres  (total 40)
  page 3 : 20 livres  (total 60)
collecte : 60 livres
